In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
files.upload()  # Select your kaggle.json file here


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sivaramthota","key":"e65c198bd4846f587eb2c64f4f2ccf4b"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d priyaadharshinivs062/leukemia-dataset


Dataset URL: https://www.kaggle.com/datasets/priyaadharshinivs062/leukemia-dataset
License(s): unknown
100% 22.3G/22.3G [03:39<00:00, 109MB/s]



In [ ]:
import zipfile
import os

# Unzip the downloaded file
with zipfile.ZipFile('leukemia-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/leukemia_data')

print("Data unzipped into /content/leukemia_data")


Data unzipped into /content/leukemia_data


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define Paths (Adjust folder names if they are different in your sidebar)
base_dir = '/content/leukemia_data'

# 2. Data Augmentation (This makes your model smarter)
datagen = ImageDataGenerator(
    rescale=1./255,            # Normalize pixels to 0-1
    validation_split=0.2,      # Use 20% for testing, 80% for training
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# 3. Create Training Data
train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),    # ResNet50 requires 224x224
    batch_size=32,
    class_mode='categorical',  # For multiple types of leukemia
    subset='training'
)

# 4. Create Validation Data
validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Print the classes found
print("Classes found:", train_generator.class_indices)


Found 16000 images belonging to 2 classes.
Found 4000 images belonging to 2 classes.
Classes found: {'test': 0, 'train': 1}


In [ ]:
import os
print(os.listdir('/content/leukemia_data/train'))


['train']


In [ ]:
# Updated path to point to the actual training images
train_dir = '/content/leukemia_data/train'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary', # Use 'binary' since you have 2 classes
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

print("Actual Classes:", train_generator.class_indices)


Found 12000 images belonging to 1 classes.
Found 3000 images belonging to 1 classes.
Actual Classes: {'train': 0}


In [ ]:
import os

# Let's peek inside to find where the classes are
path1 = '/content/leukemia_data/train'
print("Contents of /train:", os.listdir(path1))

# If there is another 'train' folder inside, check that too
path2 = '/content/leukemia_data/train/train'
if os.path.exists(path2):
    print("Contents of /train/train:", os.listdir(path2))


Contents of /train: ['train']
Contents of /train/train: ['cll train', 'all train', 'cml', 'aml train', 'h train']


In [ ]:
train_dir = '/content/leukemia_data/train/train'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical', # Changed to categorical for 5 classes
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

print("Correct Classes found:", train_generator.class_indices)


Found 12000 images belonging to 5 classes.
Found 3000 images belonging to 5 classes.
Correct Classes found: {'all train': 0, 'aml train': 1, 'cll train': 2, 'cml': 3, 'h train': 4}


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax') # Changed to 5 classes and 'softmax'
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Model is ready for Multi-class training!")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Model is ready for Multi-class training!


In [ ]:
import tensorflow as tf
device_name = tf.test.gpu_device_name()

if device_name != '/device:GPU:0':
  print('GPU NOT FOUND! Go to Runtime > Change runtime type and select T4 GPU.')
else:
  print('SUCCESS: Found GPU at: {}'.format(device_name))


SUCCESS: Found GPU at: /device:GPU:0


In [ ]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

# Save it
model.save('/content/drive/MyDrive/blood_cancer_multiclass.h5')
print("Multi-class model saved!")


Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1484s 4s/step - accuracy: 0.3727 - loss: 1.3917 - val_accuracy: 0.4377 - val_loss: 1.1454
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1532s 4s/step - accuracy: 0.4271 - loss: 1.2064 - val_accuracy: 0.4367 - val_loss: 1.1006
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1373s 4s/step - accuracy: 0.4515 - loss: 1.1591 - val_accuracy: 0.4467 - val_loss: 1.0858
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1445s 4s/step - accuracy: 0.4673 - loss: 1.1298 - val_accuracy: 0.4857 - val_loss: 1.0535
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1348s 4s/step - accuracy: 0.4737 - loss: 1.1118 - val_accuracy: 0.5373 - val_loss: 1.0412
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1362s 4s/step - accuracy: 0.4810 - loss: 1.0998 - val_accuracy: 0.4930 - val_loss: 1.0532
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1412s 4s/step - accuracy: 0.4932 - loss: 1.0778 - val_accuracy: 0.5213 - val_loss: 1.0581
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 1459s 4s/step - accuracy: 0.4978 - loss: 1.0744 - 

Multi-class model saved!
